# AMPS I · Week 03 — HW3: Implement a cell-node map
**Due:** before the Week 4 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Proofs (20 pts)

**(a) (8)** Let $V_h$ be the global P$k$ Lagrange space on a triangulation, built so that neighbouring cells share the DOFs
of their common vertices and edges. Prove that every $u\in V_h$ is continuous. Use the continuity lemma and the
$C^0$ geometric decomposition; say exactly which nodes you use on a shared edge.

**(b) (6)** Prove $\dim V_h = n_V+(k-1)\,n_E+\tfrac{(k-1)(k-2)}{2}\,n_T$ for P$k$, and deduce $\dim V_h=(kN+1)^2$ for the
$N\times N$ "right" mesh ($n_V=(N+1)^2$, $n_E=3N^2+2N$, $n_T=2N^2$).

**(c) (6)** Explain why the orientation of an edge never matters for P1 and P2, but does for P3.
For P4 on a triangle, how many DOFs sit on each edge and inside each cell, and which of them need care?

> *your proofs*

## Q2 — Your own cell-node map (40 pts)
The given cell builds a 4×4 mesh **through `create_mesh` with randomly permuted local vertex order**, as a mesh read from a file
would be. It saves the cell→vertex table `c2v`, the vertex coordinates of every cell `X` (shape `(n_cells, 3, 2)`),
the Basix node coordinates of P1–P3 (equispaced), and, for comparison, DOLFINx's DOF count and DOF coordinates.
**Do not change it.**

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import basix, basix.ufl
from dolfinx import fem, mesh

rng = np.random.default_rng(3)
N = 4
xs = np.linspace(0, 1, N + 1)
pts = np.array([(x, y) for y in xs for x in xs])
cells = []
for j in range(N):
    for i in range(N):
        v0 = j * (N + 1) + i
        cells += [[v0, v0 + 1, v0 + N + 2], [v0, v0 + N + 2, v0 + N + 1]]
cells = np.array([rng.permutation(c) for c in cells], dtype=np.int64)
msh = mesh.create_mesh(MPI.COMM_WORLD, cells, basix.ufl.element("Lagrange", "triangle", 1, shape=(2,)), pts)

msh.topology.create_entities(1)
out = dict(c2v=msh.topology.connectivity(2, 0).array.reshape(-1, 3),
           X=msh.geometry.x[:, :2][msh.geometry.dofmaps[0]],
           n_edges=msh.topology.index_map(1).size_local)
for k in (1, 2, 3):
    e = basix.create_element(basix.ElementFamily.P, basix.CellType.triangle, k, basix.LagrangeVariant.equispaced)
    V = fem.functionspace(msh, basix.ufl.element("Lagrange", "triangle", k, lagrange_variant=basix.LagrangeVariant.equispaced))
    out[f"ref{k}"] = e.points
    out[f"ndofs{k}"] = V.dofmap.index_map.size_global
    out[f"xy{k}"] = V.tabulate_dof_coordinates()[:, :2]
np.savez("/tmp/fnx/hw3_mesh.npz", **out)
print("cells", len(out["c2v"]), "| first cells (local vertex order is random):", out["c2v"][:3].tolist())

Work in ordinary cells (NumPy only). Local edge $i$ joins the local vertices `LOCAL_EDGES[i]` = `(1, 2), (0, 2), (0, 1)` (Basix);
the local DOF order is Basix's: 3 vertex DOFs, then the edge DOFs of edges 0, 1, 2, then interior DOFs.
The edge DOFs of local edge $(a,b)$ run from local vertex $a$ towards local vertex $b$.

**(a) (10)** Write `build_edges(c2v)` returning `edges` (shape `(n_E, 2)`, each row sorted) and `c2e` (shape `(n_cells, 3)`,
`c2e[c, i]` = global number of local edge $i$). Check: $n_E$ equals DOLFINx's `n_edges`, and $n_V-n_E+n_T=1$.

**(b) (15)** Write `cell_node_map(c2v, c2e, nV, nE, k, orient=True)` for $k=1,2,3$, numbering vertex DOFs first, then edge DOFs,
then interior DOFs (lecture §C1). With `orient=True`, order every edge's DOFs from its lower to its higher **global** vertex.

**(c) (10)** Verify your map: for each cell, map the reference nodes `ref{k}` to the cell ($x=x_0+J\,X$) and write the coordinates
into a global array at `M[c]`. Report (i) the largest disagreement between two cells writing the same DOF, (ii) your DOF count vs
DOLFINx's, (iii) whether your set of DOF coordinates equals DOLFINx's `xy{k}` (DOLFINx numbers them differently; compare as sets).
Plot the mesh with your P3 DOF numbers.

**(d) (5)** Run the check of (c) with `orient=False` for $k=2$ and $k=3$. Report the disagreement and explain it (link to Q1(c)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load("/tmp/fnx/hw3_mesh.npz")
LOCAL_EDGES = [(1, 2), (0, 2), (0, 1)]
# TODO: (a) build_edges(c2v)  (b) cell_node_map(c2v, c2e, nV, nE, k, orient=True)
# TODO: (c) checks (i)–(iii) for k = 1, 2, 3 and the P3 plot  (d) orient=False for k = 2, 3

> *(d) your explanation*

## Q3 — CG vs DG and the parallel index map (15 pts)
Run on **2 ranks**. For CG P2 and DG P2 on an 8×8 mesh, print on each rank: owned DOFs, ghost DOFs, `size_global`,
and the sum of owned DOFs over ranks (`allreduce`). Then answer:
(i) Is $\sum_{\rm ranks}(\text{owned}+\text{ghost})$ equal to `size_global`? Why (not)?
(ii) Why does the DG space have ghost DOFs at all?
(iii) Which number would you report as "the number of unknowns" in a paper, and how do you get it in code that runs on any number of ranks?

In [ ]:
%%fenicsx -np 2
# TODO: CG P2 and DG P2 on an 8×8 mesh; per-rank owned / ghost / size_global; allreduce of owned

> *your answers to (i)–(iii)*

## Q4 — LLM-generated code with planted bugs (25 pts)
A student asked an LLM: *"DOLFINx code, run on 2 ranks: for an 8×8 unit square, print the number of cells, the total area
(from the vertex coordinates), the number of boundary facets, and the number of scalar DOFs of a vector P2 space."*
The answer below is pasted **unchanged**. It contains several bugs; some crash, some silently give wrong numbers.
Run it (the crash is expected).

**(a) (15)** Fix the code in the next cell, one bug at a time. For each bug write: the line, what goes wrong (error or wrong value),
how you detected it, and the fix. *(Hint: there are at least five.)*

**(b) (10)** Verify the fixed code against answers you can compute by hand, and show that it prints the same numbers with `-np 1` and `-np 2`.

In [ ]:
%%fenicsx -np 2
# EXPECTED-FAIL — LLM output, unchanged
from mpi4py import MPI
import numpy as np
from dolfinx import fem, mesh

comm = MPI.COMM_WORLD
msh = mesh.create_unit_square(comm, 8, 8)
x = msh.geometry.x

num_cells = msh.topology.index_map(2).size_local
area = 0.0
for c in range(msh.geometry.dofmap.shape[0]):
    v = x[msh.geometry.dofmap.links(c)]
    J = np.array([[v[1, 0] - v[0, 0], v[2, 0] - v[0, 0]],
                  [v[1, 1] - v[0, 1], v[2, 1] - v[0, 1]]])
    area += np.linalg.det(J) / 2

boundary = mesh.exterior_facet_indices(msh.topology)

V = fem.functionspace(msh, ("Lagrange", 2, (2,)))
ndofs = V.dofmap.index_map.size_global

print(f"cells {num_cells}, area {area:.6f}, boundary facets {len(boundary)}, vector P2 DOFs {ndofs}")

In [ ]:
%%fenicsx -np 2
# TODO: fixed version

> *(a) your bug list and (b) your verification*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …